<a href="https://colab.research.google.com/github/VladMursalimov/swagathon/blob/develop/EmbeddigsClusterization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install sentence-transformers numpy pandas


In [3]:
import pandas as pd

In [4]:
from sentence_transformers import SentenceTransformer

In [5]:
model = SentenceTransformer("ai-forever/FRIDA")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [21]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

df = pd.read_excel("./sample_data/data.xlsx", 'Лист1')

df = df.head(100)

# 2. Подготовка данных
# Список колонок, которые нужно соединить в одну строку для анализа
columns_to_join = ['название сте', 'модель', 'производитель', 'название категории', 'характеристики']
#columns_to_join = ['название сте']

# Проверяем, есть ли эти колонки в файле, чтобы избежать ошибок
missing_cols = [col for col in columns_to_join if col not in df.columns]
if missing_cols:
    raise ValueError(f"В файле не найдены колонки: {missing_cols}")

# Заполняем пустые значения, приводим всё к строке и объединяем через пробел
df['combined_text'] = df[columns_to_join].fillna('').astype(str).agg(' '.join, axis=1)

# Очистка от лишних пробелов (опционально, но полезно)
df['combined_text'] = df['combined_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [22]:
# 3. Векторизация
# Используем SBERT для русского языка (лучше всего подходит для Entity Resolution)
model = SentenceTransformer("ai-forever/FRIDA")

embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

# Нормализация векторов (важно для корректной работы косинусного расстояния)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [23]:
# 4. Кластеризация (Entity Resolution)
# AgglomerativeClustering отлично подходит, когда мы не знаем число кластеров,
# но знаем, насколько похожими должны быть дубликаты.

# distance_threshold - порог схожести (0.0 - полное совпадение, 1.0 - полная разница).
# 0.15 - 0.25 обычно хороший диапазон для SBERT.
threshold = 0.1

print(f"Запуск кластеризации с порогом {threshold}...")
cluster_model = AgglomerativeClustering(
    n_clusters=None,             # Автоматическое определение количества групп
    distance_threshold=threshold,
    metric='cosine',             # Используем косинусное расстояние
    linkage='average'            # Метод связывания average дает стабильные результаты
)

# Присваиваем каждому товару ID его группы (кластера)
df['group_id'] = cluster_model.fit_predict(embeddings)

# 5. Сохранение результата
output_file = 'result_with_groups.xlsx'

# Сортируем по group_id, чтобы дубликаты шли друг за другом в Excel
df_sorted = df.sort_values(by='group_id')

df_sorted.to_excel(output_file, index=False)

# Вывод небольшого примера в консоль
print("\nПример найденных групп:")
example_groups = df_sorted['group_id'].value_counts().head(3).index # Берем топ-3 крупных кластера
print(df_sorted[df_sorted['group_id'].isin(example_groups)][['group_id', 'combined_text']].head(10))

Запуск кластеризации с порогом 0.1...

Пример найденных групп:
    group_id                                      combined_text
12         0  Автошина 11.00 R22.5 КАМА NF-701 КАМА NF-701 О...
16         0  Шины Кама NF701 NF701 ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБ...
36         1  Шина пневматическая (с камерой) WORK MASTER R-...
34         1  Шина пневматическая (с камерой) 10.0/75-15.3 1...
54         2  Шина 12,00R20 ИД-304 ИД-304 ПУБЛИЧНОЕ АКЦИОНЕР...
62         2  Шина 12,00R20 ИД-304 (18 нс) ОШЗ ИД-304 (18нс)...
